In [5]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/03/cfz_fs5x4hj569n9qp0w2nxh0000gn/T/pip-install-ewh1t4zz/polara_cf04be96031e448ab12503db7429cf88
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/03/cfz_fs5x4hj569n9qp0w2nxh0000gn/T/pip-install-ewh1t4zz/polara_cf04be96031e448ab12503db7429cf88
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from typing import Callable, Dict, List, Tuple, Any, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda_lag,load_yambda, split_and_reindex)

In [89]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": None,
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ratings_Books.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [90]:
# ml_df = load_ml20m(config["ml-20m"]["interactions_path"], config=config["ml-20m"])
yambda_df_retrieval = load_yambda(interactions_path=config["yambda-retrieval"]["interactions_path"], config=config["yambda-retrieval"])
# yambda_df = load_yambda_lag(interactions_path=None, config=config["yambda-retrieval"])
# amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [91]:
# ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml-20m"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df_retrieval, config=config["yambda-retrieval"])
# amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [92]:
yambda_train

,user_id,item_id,timestamp,feedback
0,0,120737,15929455,1
1,1,111939,18111700,1
2,1,162955,18111700,1
3,1,121703,18111945,1
4,2,161375,106910,1
...,...,...,...,...
9033955,79057,37399,20521405,1
9033956,79058,64414,14299485,1
9033957,79058,123244,19336945,1
9033958,79058,77874,19337350,1


In [93]:
yambda_test["timestamp"].min()

np.uint32(24223920)

In [94]:
yambda_df_retrieval

,user_id,item_id,timestamp,feedback
0,10,6928395,15929455,1
1,20,6417612,18111700,1
2,20,9362291,18111700,1
3,20,6983647,18111945,1
4,30,9270848,106910,1
...,...,...,...,...
9033955,999990,2146481,20521405,1
9033956,1000000,3694666,14299485,1
9033957,1000000,7071820,19336945,1
9033958,1000000,4465732,19337350,1


In [95]:
datasets = {
    # "ml-20m": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda-retrieval": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [96]:
for ds in datasets.values():
    display(ds['train'].head())

,user_id,item_id,timestamp,feedback
0,0,120737,15929455,1
1,1,111939,18111700,1
2,1,162955,18111700,1
3,1,121703,18111945,1
4,2,161375,106910,1


In [97]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda-retrieval,7473676,748537,8222213,0.909,0.091,79059,53927,163448


## HSTU Amazon Books experiment

Запуск HSTU-пайплайна на Amazon Books

In [98]:
import torch

from src.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.models.hstu import HSTUModel


In [26]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["amzn-books"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=4,
    num_heads=4,
    linear_dim=16,
    attention_dim=16,
    input_dropout_rate=0.5,
    linear_dropout_rate=0.5,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=amzn_train,
    test=amzn_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=amzn_data_description["users"],
    item_col=amzn_data_description["items"],
    time_col=amzn_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


(1161643, 215463, 215463)

In [28]:
hstu_model = HSTUModel(
    num_items=amzn_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [29]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=57,606,144, trainable=57,606,144
pos_embeddings: total=3,200, trainable=3,200
input_dropout: total=0, trainable=0
blocks: total=84,112, trainable=84,112

params_sum=57,693,456, trainable_sum=57,693,456


In [46]:
losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
hstu_losses

KeyboardInterrupt: 

In [18]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


HSTU evaluation: 100%|██████████| 1684/1684 [02:12<00:00, 12.68it/s]


{'hitrate': 0.02734576238147617,
 'recall': 0.012883055933276683,
 'ndcg': 0.0033107533984527526,
 'coverage': 0.01155766891272588}

## HSTU Yambda experiment

Запуск HSTU-пайплайна на Yambda

In [99]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["yambda-retrieval"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=2,
    num_heads=2,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


(121756, 53927, 53927)

In [100]:
hstu_model = HSTUModel(
    num_items=yambda_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [101]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=10,460,736, trainable=10,460,736
pos_embeddings: total=6,400, trainable=6,400
input_dropout: total=0, trainable=0
blocks: total=83,472, trainable=83,472

params_sum=10,550,608, trainable_sum=10,550,608


In [74]:
yambda_data_description

{'users': 'user_id',
 'items': 'item_id',
 'timestamp': 'timestamp',
 'feedback': 'feedback',
 'n_users': 68140,
 'n_items': 163434}

In [75]:
losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
losses

epoch 1/10:   0%|          | 0/906 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

In [ ]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

In [ ]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


In [ ]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics